In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

# Display versions to verify environment
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

# Verify file pathing for your data directory
data_dir = Path("data/processed")
print(f"Data directory status: {'Exists' if data_dir.exists() else 'Needs creation'}")

pandas version: 2.2.2
NumPy version: 2.0.2
Data directory status: Needs creation


In [4]:
from pathlib import Path

# Set the project root path
PROJECT_ROOT = Path("/content/drive/MyDrive/GSSRP_2026/student_performance_project")

# Define the path to the processed data folder
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create the folder if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Print status to verify setup
print("Project root:", PROJECT_ROOT)
print("Processed-data folder:", PROCESSED_DIR)
print("Folder exists:", PROCESSED_DIR.exists())

Project root: /content/drive/MyDrive/GSSRP_2026/student_performance_project
Processed-data folder: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed
Folder exists: True


In [8]:
import pandas as pd
from pathlib import Path

# 1. Define the correct path
# If your student-mat.csv is in the project root, this will find it:
file_path = "student-mat.csv"

# 2. Safety Check: Verify the file exists before loading
if Path(file_path).exists():
    # Note: Use sep=";" because the CSV files you uploaded use semicolons
    df = pd.read_csv(file_path, sep=";")
    print("Successfully loaded:", file_path)
    print("Current DataFrame shape:", df.shape)
    display(df.head())
else:
    print(f"Error: Could not find '{file_path}'. Check your working directory.")
    # Debug: see where we are currently looking
    print("Current working directory:", Path.cwd())

Successfully loaded: student-mat.csv
Current DataFrame shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [9]:
# Identify categorical columns
cat_cols = df.select_dtypes(include="object").columns

# Encode
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)
df_encoded = df_encoded.astype(int)

# Verify
print("New encoded shape:", df_encoded.shape)

New encoded shape: (395, 42)


In [11]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

Number of rows: 395
Number of columns: 33

Column names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


In [12]:
def inspect_dtypes(dataframe):
    """Creates a readable DataFrame of column types."""
    dtype_table = pd.DataFrame(
        {
            "column": dataframe.columns,
            "dtype": dataframe.dtypes.astype(str).values
        }
    )
    return dtype_table

# Use it to see types before encoding
print("Types before encoding:")
display(inspect_dtypes(df))

Types before encoding:


,column,dtype
0,school,object
1,sex,object
2,age,int64
3,address,object
4,famsize,object
5,Pstatus,object
6,Medu,int64
7,Fedu,int64
8,Mjob,object
9,Fjob,object


In [13]:
# Create a summary table of missing values
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

# Calculate the percentage of missing values per column
missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

# Display the top 20 columns with the most missing data
display(missing_summary.head(20))

,missing_count,missing_percent
school,0,0.0
sex,0,0.0
age,0,0.0
address,0,0.0
famsize,0,0.0
Pstatus,0,0.0
Medu,0,0.0
Fedu,0,0.0
Mjob,0,0.0
Fjob,0,0.0


In [14]:
# Check if categorical columns were identified
if len(cat_cols) == 0:
    raise ValueError(
        "No object-type categorical columns were found. "
        "The dataset may already be encoded, or the categorical "
        "columns may use a different data type."
    )

print(f"Categorical columns found: {list(cat_cols)}")
print("Encoding can continue.")

Categorical columns found: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Encoding can continue.


In [15]:
# Check if categorical columns were identified
if len(cat_cols) == 0:
    raise ValueError(
        "No object-type categorical columns were found. "
        "The dataset may already be encoded, or the categorical "
        "columns may use a different data type."
    )

print(f"Categorical columns found: {list(cat_cols)}")
print("Categorical columns were found. Encoding can continue.")

Categorical columns found: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Categorical columns were found. Encoding can continue.


In [16]:
# Audit categorical columns before encoding
for column in cat_cols:
    unique_values = df[column].drop_duplicates().tolist()

    print("=" * 70)
    print("Column:", column)
    print("Number of nonmissing categories:", df[column].nunique(dropna=True))
    print("Number of missing values:", df[column].isna().sum())
    print("Example categories:", unique_values[:10])

Column: school
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GP', 'MS']
Column: sex
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['F', 'M']
Column: address
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['U', 'R']
Column: famsize
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GT3', 'LE3']
Column: Pstatus
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['A', 'T']
Column: Mjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['at_home', 'health', 'other', 'services', 'teacher']
Column: Fjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['teacher', 'other', 'services', 'health', 'at_home']
Column: reason
Number of nonmissing categories: 4
Number of missing values: 0
Example categories: ['course', 'other', 'home', 'reputation']
Column: g

In [17]:
# Identify columns with high cardinality that might be unique identifiers
possible_identifier_columns = []

for column in cat_cols:
    uniqueness_ratio = df[column].nunique(dropna=False) / len(df)

    # Flag columns where > 90% of values are unique
    if uniqueness_ratio > 0.90:
        possible_identifier_columns.append(
            {
                "column": column,
                "unique_values": df[column].nunique(dropna=False),
                "uniqueness_ratio": round(uniqueness_ratio, 3)
            }
        )

# Review findings
if possible_identifier_columns:
    print("Review these possible identifier columns before encoding:")
    display(pd.DataFrame(possible_identifier_columns))
else:
    print("No likely identifier columns were detected.")

No likely identifier columns were detected.


In [18]:
rows_before = df.shape[0]
columns_before = df.shape[1]
print("Rows before encoding:", rows_before)
print("Columns before encoding:", columns_before)

Rows before encoding: 395
Columns before encoding: 33


In [19]:
# Calculate expected expansion for one-hot encoding
category_count_table = []

for column in cat_cols:
    category_count = df[column].nunique(dropna=True)
    # With drop_first=True, we expect k-1 new columns
    dummy_count = max(category_count - 1, 0)

    category_count_table.append(
        {
            "original_column": column,
            "number_of_categories": category_count,
            "expected_dummy_columns": dummy_count
        }
    )

# Convert to DataFrame for a clean view
category_count_df = pd.DataFrame(category_count_table)
display(category_count_df)

# Total expected columns check
print(f"Total dummy columns to be added: {category_count_df['expected_dummy_columns'].sum()}")

,original_column,number_of_categories,expected_dummy_columns
0,school,2,1
1,sex,2,1
2,address,2,1
3,famsize,2,1
4,Pstatus,2,1
5,Mjob,5,4
6,Fjob,5,4
7,reason,4,3
8,guardian,3,2
9,schoolsup,2,1


Total dummy columns to be added: 26


In [20]:
# Calculate total expected columns
number_of_numeric_columns = df.shape[1] - len(cat_cols)
expected_dummy_columns = category_count_df["expected_dummy_columns"].sum()
expected_total_columns = number_of_numeric_columns + expected_dummy_columns

print("Original numeric columns:", number_of_numeric_columns)
print("Expected dummy columns:", expected_dummy_columns)
print("Expected final column count:", expected_total_columns)

Original numeric columns: 16
Expected dummy columns: 26
Expected final column count: 42


In [21]:
# One-hot encode nominal categorical columns
cat_cols = df.select_dtypes(include="object").columns

# Apply one-hot encoding with drop_first=True to avoid the dummy-variable trap
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)

# Print the resulting shape
print("Encoded shape:", df_encoded.shape)

Encoded shape: (395, 42)


In [22]:
# Convert categorical features into binary/dummy variables
df_encoded = pd.get_dummies(
    df,
    columns=list(cat_cols),
    drop_first=True
)

# Optional: Convert True/False to 1/0 for cleaner numerical representation
df_encoded = df_encoded.astype(int)

# Verify the result
print("New DataFrame shape:", df_encoded.shape)

New DataFrame shape: (395, 42)


In [23]:
rows_after = df_encoded.shape[0]
columns_after = df_encoded.shape[1]
print("Rows before encoding:", rows_before)
print("Rows after encoding:", rows_after)
print("\nColumns before encoding:", columns_before)
print("Columns after encoding:", columns_after)
print("\nNet change in columns:", columns_after - columns_before)

Rows before encoding: 395
Rows after encoding: 395

Columns before encoding: 33
Columns after encoding: 42

Net change in columns: 9


In [24]:
# Assuming columns_after was calculated using: columns_after = df_encoded.shape[1]

print("Expected final column count:", expected_total_columns)
print("Actual final column count:", columns_after)

if columns_after == expected_total_columns:
    print("Column-count check passed.")
else:
    print(
        "Column-count check requires review. "
        "Missing values or unusual category structures may affect the result."
    )

Expected final column count: 42
Actual final column count: 42
Column-count check passed.


In [25]:
# Identify columns removed and added during encoding
original_columns = set(df.columns)
encoded_columns = set(df_encoded.columns)

# Find columns that were transformed (removed from the original set)
removed_columns = [
    column for column in df.columns
    if column not in encoded_columns
]

# Find the new binary dummy columns created
new_columns = [
    column for column in df_encoded.columns
    if column not in original_columns
]

print("Original categorical columns removed:")
print(removed_columns)
print("\nNumber of new dummy columns:", len(new_columns))
print("\nFirst 30 new dummy columns:")
print(new_columns[:30])

Original categorical columns removed:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of new dummy columns: 26

First 30 new dummy columns:
['school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'Pstatus_T', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other', 'schoolsup_yes', 'famsup_yes', 'paid_yes', 'activities_yes', 'nursery_yes', 'higher_yes', 'internet_yes', 'romantic_yes']


In [26]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [27]:
display(df_encoded[new_columns].head())

,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,0,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,0,1,1,0,0
1,0,0,1,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,1,1,0
2,0,0,1,1,1,0,0,0,0,0,...,1,0,1,0,1,0,1,1,1,0
3,0,0,1,0,1,1,0,0,0,0,...,1,0,0,1,1,1,1,1,1,1
4,0,0,1,0,1,0,1,0,0,0,...,0,0,0,1,1,0,1,1,0,0


In [29]:
boolean_columns = df_encoded.select_dtypes(include="bool").columns

In [30]:
print("Number of Boolean dummy columns:", len(boolean_columns))

Number of Boolean dummy columns: 0


In [31]:
# Identify which columns were created by get_dummies (if you haven't already)
# Alternatively, you can convert all columns that only contain 0s and 1s
boolean_columns = [
    col for col in df_encoded.columns
    if df_encoded[col].isin([0, 1]).all()
]

# Convert these columns to the most memory-efficient integer type
df_encoded[boolean_columns] = df_encoded[boolean_columns].astype("int8")

print("Boolean dummy variables converted to 0 and 1 (int8 type).")
print(f"Memory usage optimized for {len(boolean_columns)} columns.")

Boolean dummy variables converted to 0 and 1 (int8 type).
Memory usage optimized for 26 columns.


In [32]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [33]:
# Identify any remaining columns with 'object' data type
remaining_object_columns = (
    df_encoded
    .select_dtypes(include="object")
    .columns
    .tolist()
)

# Audit the result
if len(remaining_object_columns) == 0:
    print("Success: No object-type columns remain. The dataset is fully numeric.")
else:
    print("Warning: The following object-type columns still exist and may require further manual encoding:")
    print(remaining_object_columns)

Success: No object-type columns remain. The dataset is fully numeric.


In [34]:
# Final validation: Ensure no object-type columns exist
assert len(remaining_object_columns) == 0, (
    f"Some object-type columns remain after encoding: {remaining_object_columns}"
)

print("Object-column check passed. Dataset is ready for modeling.")

Object-column check passed. Dataset is ready for modeling.


In [35]:
# Final validation: Ensure the number of rows remains constant
assert df_encoded.shape[0] == df.shape[0], (
    f"The number of rows changed: {df.shape[0]} original vs {df_encoded.shape[0]} encoded."
)

print("Row-count check passed. Integrity of records maintained.")

Row-count check passed. Integrity of records maintained.


In [36]:
# Final validation: Ensure the index order and labels were preserved
assert df_encoded.index.equals(df.index), (
    "The row index changed or was reordered during encoding."
)

print("Row-index check passed. Data alignment is preserved.")

Row-index check passed. Data alignment is preserved.


In [37]:
# Validate that the target variable (G3) is preserved and unmodified
if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "The G3 target variable is missing after encoding."
    )

    # Ensure the values and order of the target are identical to the original
    pd.testing.assert_series_equal(
        df_encoded["G3"],
        df["G3"],
        check_names=True
    )
    print("G3 target-variable check passed.")
else:
    print("G3 was not found in the current dataset.")

G3 target-variable check passed.


In [38]:
# Check for any duplicate column names in the encoded DataFrame
duplicate_column_count = df_encoded.columns.duplicated().sum()

print("Duplicate column names detected:", duplicate_column_count)

# Assert that there are no duplicates to ensure clean input for modeling
assert duplicate_column_count == 0, (
    "Duplicate column names were detected. Please review your input columns."
)

print("Duplicate-column check passed. The structure is clean.")

Duplicate column names detected: 0
Duplicate-column check passed. The structure is clean.


In [39]:
missing_after = df_encoded.isna().sum().sum()
print("Total missing values before encoding:", df.isna().sum().sum())
print("Total missing values after encoding:", missing_after)

Total missing values before encoding: 0
Total missing values after encoding: 0


In [40]:
# Final check for any remaining missing values
columns_with_missing_values = (
    df_encoded.isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
)

if columns_with_missing_values.empty:
    print("No missing values remain. The dataset is clean.")
else:
    print("Warning: Columns still containing missing values:")
    display(columns_with_missing_values.to_frame("missing_count"))

No missing values remain. The dataset is clean.


In [41]:
# Consolidated sanity checks for the encoded dataset
assert df_encoded.shape[0] == df.shape[0], (
    "Sanity check failed: row count changed."
)
assert df_encoded.index.equals(df.index), (
    "Sanity check failed: row index changed."
)
assert not df_encoded.columns.duplicated().any(), (
    "Sanity check failed: duplicate column names exist."
)

# Verify all columns are now numeric
remaining_objects = df_encoded.select_dtypes(include="object").columns.tolist()
assert len(remaining_objects) == 0, (
    f"Sanity check failed: object columns remain: {remaining_objects}"
)

# Validate the target variable integrity
if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "Sanity check failed: G3 is missing."
    )
    # Ensure values, index, and data type are identical
    pd.testing.assert_series_equal(df_encoded["G3"], df["G3"])

print("All one-hot encoding sanity checks passed. The dataset is fully validated.")

All one-hot encoding sanity checks passed. The dataset is fully validated.


In [42]:
# Create a comprehensive summary report of the encoding process
encoding_summary = pd.DataFrame(
    {
        "measurement": [
            "Rows before encoding",
            "Rows after encoding",
            "Columns before encoding",
            "Columns after encoding",
            "Categorical columns encoded",
            "New dummy columns",
            "Remaining object columns",
            "Missing values after encoding"
        ],
        "value": [
            df.shape[0],
            df_encoded.shape[0],
            df.shape[1],
            df_encoded.shape[1],
            len(cat_cols),
            len(new_columns),
            len(remaining_object_columns),
            df_encoded.isna().sum().sum()
        ]
    }
)

# Display the summary for a final review
display(encoding_summary)

,measurement,value
0,Rows before encoding,395
1,Rows after encoding,395
2,Columns before encoding,33
3,Columns after encoding,42
4,Categorical columns encoded,17
5,New dummy columns,26
6,Remaining object columns,0
7,Missing values after encoding,0


In [43]:
from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Define project paths
# ------------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/GSSRP_2026/student_performance_project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "student_data_clean.csv"
OUTPUT_FILE = PROCESSED_DIR / "student_data_encoded.csv"

# ------------------------------------------------------------
# 3. Load the cleaned dataset
# ------------------------------------------------------------
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE, sep=None, engine="python")
print(f"Dataset loaded: {INPUT_FILE.name}")
print(f"Original shape: {df.shape}")

# ------------------------------------------------------------
# 4. Identify categorical columns
# ------------------------------------------------------------
cat_cols = df.select_dtypes(include="object").columns
print(f"\nCategorical columns ({len(cat_cols)}): {list(cat_cols)}")

# ------------------------------------------------------------
# 5. One-hot encode categorical columns
# ------------------------------------------------------------
# Using drop_first=True to avoid the dummy-variable trap
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)

# ------------------------------------------------------------
# 6. Convert Boolean dummy variables to integers
# ------------------------------------------------------------
boolean_columns = df_encoded.select_dtypes(include="bool").columns
df_encoded[boolean_columns] = df_encoded[boolean_columns].astype("int8")

print(f"Encoded shape: {df_encoded.shape}")

# ------------------------------------------------------------
# 7. Final Sanity Checks
# ------------------------------------------------------------
assert df_encoded.shape[0] == df.shape[0], "Row count mismatch!"
assert df_encoded.index.equals(df.index), "Row index mismatch!"
assert not df_encoded.columns.duplicated().any(), "Duplicate columns detected!"
assert len(df_encoded.select_dtypes(include="object").columns) == 0, "Object columns remain!"

if "G3" in df.columns:
    assert "G3" in df_encoded.columns, "Target variable G3 missing!"
    pd.testing.assert_series_equal(df_encoded["G3"], df["G3"])

print("All one-hot encoding sanity checks passed.")

# ------------------------------------------------------------
# 8. Save and Verify
# ------------------------------------------------------------
df_encoded.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE.name}")

# Reload to verify
df_encoded_check = pd.read_csv(OUTPUT_FILE)
assert df_encoded_check.shape == df_encoded.shape, "Reload verification failed!"
print("Saved-file verification passed.")

# Preview the results
display(df_encoded.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Input file not found: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/student_data_clean.csv

In [45]:
# Update the INPUT_FILE path here
INPUT_FILE = PROCESSED_DIR / "student-mat.csv"
# If the file is in a raw folder
INPUT_FILE = PROJECT_ROOT / "data" / "raw" / "student-mat.csv"
# Debugging check: Run this if you are still getting an error
import os
print("Does file exist?", INPUT_FILE.exists())
print("Looking in path:", INPUT_FILE)

Does file exist? False
Looking in path: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/raw/student-mat.csv


In [46]:
# --- Update this section in your script ---
# If the file is named student-mat.csv, use this:
INPUT_FILE = PROCESSED_DIR / "student-mat.csv"

# --- OR, if you have already saved it as student_data_clean.csv ---
# Ensure the file is actually in the 'processed' folder:
# INPUT_FILE = PROCESSED_DIR / "student_data_clean.csv"

In [47]:
import os

# Check if the path exists
print(f"Checking path: {INPUT_FILE}")
if INPUT_FILE.exists():
    print("✅ File found! Proceeding to load.")
else:
    print("❌ File NOT found. Please check your folder structure.")
    # This lists the files in your processed directory so you can see the correct name
    print("Files in directory:", list(PROCESSED_DIR.iterdir()))

Checking path: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/student-mat.csv
❌ File NOT found. Please check your folder structure.
Files in directory: []


In [48]:
# ------------------------------------------------------------
# 3. Load the cleaned dataset (Fixed Version)
# ------------------------------------------------------------
import os

# Define the target file name
target_filename = "student-mat.csv"
INPUT_FILE = PROCESSED_DIR / target_filename

# Debugging: List files if it's not found
if not INPUT_FILE.exists():
    available_files = [f.name for f in PROCESSED_DIR.iterdir()]
    raise FileNotFoundError(
        f"Could not find '{target_filename}' in {PROCESSED_DIR}.\n"
        f"Available files are: {available_files}\n"
        "Please check if the file name is different."
    )

df = pd.read_csv(INPUT_FILE, sep=None, engine="python")
print(f"✅ Dataset loaded successfully: {INPUT_FILE.name}")
print(f"Original shape: {df.shape}")

FileNotFoundError: Could not find 'student-mat.csv' in /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed.
Available files are: []
Please check if the file name is different.

In [49]:
# Search for all CSV files in your project directory
found_files = list(PROJECT_ROOT.rglob("*.csv"))
print("Found these CSV files in your project:")
for f in found_files:
    print(f)

Found these CSV files in your project:


In [50]:
# Change this line based on what the search found above
INPUT_FILE = PROJECT_ROOT / "data" / "raw" / "student-mat.csv"

In [51]:
!find /content/drive/MyDrive/GSSRP_2026/student_performance_project -name "*.csv"

In [53]:
import os
for root, dirs, files in os.walk("/content/drive/MyDrive/GSSRP_2026"):
    for file in files:
        if "student" in file and file.endswith(".csv"):
            print(os.path.join(root, file))

In [55]:
from pathlib import Path
import pandas as pd

# Use this path AFTER you have dragged and dropped the file into the sidebar
INPUT_FILE = Path("/content/student-mat.csv")

# Load Dataset
df = pd.read_csv(INPUT_FILE, sep=None, engine="python")
print(f"✅ Loaded: {df.shape}")

# Proceed with your logic
cat_cols = df.select_dtypes(include="object").columns
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)
bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype("int8")

# Display first few rows to confirm success
display(df_encoded.head())

✅ Loaded: (395, 33)


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [56]:
# 1. Define X_full (features) and y_full (target)
# We drop G3 from the features because it is our target variable (preventing leakage)
X_full = df_encoded.drop(columns=["G3"])
y_full = df_encoded["G3"]

print(f"Full-information features shape: {X_full.shape}")
print(f"Target variable shape: {y_full.shape}")

# 2. Validation Checks
assert "G3" not in X_full.columns, "Error: G3 still in features!"
assert "G1" in X_full.columns and "G2" in X_full.columns, "Error: G1/G2 missing!"
assert X_full.shape[0] == y_full.shape[0], "Error: Row mismatch!"

# 3. Install necessary library for Parquet
!pip install pyarrow

# 4. Save the files
# Saving as both Parquet (efficient) and CSV (readable)
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y_full.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

X_full.to_csv(PROCESSED_DIR / "X_full.csv", index=False)
y_full.to_csv(PROCESSED_DIR / "y_full.csv", index=False)
df_encoded.to_csv(PROCESSED_DIR / "full_data.csv", index=False)

print("\n✅ Session 20: Files saved to 'data/processed/'")

Full-information features shape: (395, 41)
Target variable shape: (395,)

✅ Session 20: Files saved to 'data/processed/'


In [57]:
X_full = df_encoded.drop(columns=["G3"])
y = df_encoded["G3"]
print("Full-information features:", X_full.shape[1])

Full-information features: 41


# GSSRP 2026 — Session 20
## Feature Engineering II: Full-Information Feature Set
This notebook creates the full-information modeling dataset. The predictors include
G1 and G2, while G3 is separated as the target variable.

In [58]:
import pandas as pd
print("Pandas version:", pd.__version__)

Pandas version: 2.2.2


In [60]:
import pandas as pd
from pathlib import Path

# 1. Ensure df_encoded is ready
if "df_encoded" not in globals():
    # Adjust path if needed; use the one we established in Session 19
    INPUT_FILE = Path("/content/student_data_encoded.csv")
    df_encoded = pd.read_csv(INPUT_FILE)
    print("✅ Loaded df_encoded from file.")
else:
    print("✅ df_encoded is already in memory.")

# 2. Assemble Full-Information Sets
X_full = df_encoded.drop(columns=["G3"])
y_full = df_encoded["G3"]

print(f"Full-information features shape: {X_full.shape}")
print(f"Target variable shape: {y_full.shape}")

# 3. Required Validation (Per Session 20 Checklist)
assert "G3" not in X_full.columns, "❌ Error: G3 still in features!"
assert "G1" in X_full.columns and "G2" in X_full.columns, "✅ G1 and G2 present."
assert X_full.shape[0] == y_full.shape[0], "❌ Error: Row mismatch!"

# 4. Save to processed directory (as per requirements)
PROCESSED_DIR = Path("/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y_full.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

print("\n✅ Session 20: Full-Information dataset assembled and saved.")

✅ df_encoded is already in memory.
Full-information features shape: (395, 41)
Target variable shape: (395,)

✅ Session 20: Full-Information dataset assembled and saved.


In [61]:
import pandas as pd

# ------------------------------------------------------------
# 1. Validate Input DataFrame
# ------------------------------------------------------------
if "df_encoded" not in globals():
    raise NameError("df_encoded is not defined. Load your dataset first.")

required_columns = ["G1", "G2", "G3"]
missing = [c for c in required_columns if c not in df_encoded.columns]
if missing:
    raise KeyError(f"Required columns missing: {', '.join(missing)}")

# ------------------------------------------------------------
# 2. Assemble Full-Information Features and Target
# ------------------------------------------------------------
X_full = df_encoded.drop(columns=["G3"]).copy()
y = df_encoded["G3"].copy()

# ------------------------------------------------------------
# 3. Validation Logic (Checklist Compliance)
# ------------------------------------------------------------
assert "G3" not in X_full.columns, "Target leakage detected: G3 in X_full."
assert "G1" in X_full.columns and "G2" in X_full.columns, "G1/G2 missing from X_full."
assert len(X_full) == len(y), "Row count mismatch between X_full and y."
assert X_full.index.equals(y.index), "Index mismatch between X_full and y."

# ------------------------------------------------------------
# 4. Results Summary
# ------------------------------------------------------------
print("=" * 60)
print("SESSION 20: FULL-INFORMATION DATASET SUMMARY")
print("=" * 60)
print(f"Features shape: {X_full.shape}")
print(f"Target shape: {y.shape}")
print(f"G1 & G2 Included: {'G1' in X_full.columns and 'G2' in X_full.columns}")
print(f"G3 Excluded from X: {'G3' not in X_full.columns}")
print(f"Missing Values: {X_full.isna().sum().sum() + y.isna().sum()}")
print("=" * 60)
print("✅ Verification passed: Full-information dataset is ready.")

SESSION 20: FULL-INFORMATION DATASET SUMMARY
Features shape: (395, 41)
Target shape: (395,)
G1 & G2 Included: True
G3 Excluded from X: True
Missing Values: 0
✅ Verification passed: Full-information dataset is ready.


In [62]:
# 1. Validation: Target Exclusion
assert "G3" not in X_full.columns, "❌ Target leakage detected: G3 is still present in X_full."
print("✅ Verification passed: G3 is correctly excluded from the feature matrix.")

# 2. Validation: Inclusion of Required Predictors
assert "G1" in X_full.columns and "G2" in X_full.columns, "❌ G1 or G2 missing from X_full."
print("✅ Verification passed: G1 and G2 are present in the feature matrix.")

# 3. Validation: Dimensional Integrity
assert len(X_full) == len(y), "❌ Row count mismatch between features and target."
print("✅ Verification passed: Row counts match.")

✅ Verification passed: G3 is correctly excluded from the feature matrix.
✅ Verification passed: G1 and G2 are present in the feature matrix.
✅ Verification passed: Row counts match.


### Summary

The full-information feature set ($X_{full}$) has been successfully decoupled from the target variable ($y$). The feature matrix now contains all necessary predictor variables, including $G1$ and $G2$, while the target vector ($y$) exclusively holds the $G3$ final grades. All integrity checks—including row count matching, index alignment, and the exclusion of the target—have passed.

### Interpretation

* **Target Exclusion:** The removal of `G3` from the feature matrix is verified. This ensures no data leakage occurs, preventing the model from having "advance knowledge" of the outcome during the training phase.
* **Feature Count:** Your feature matrix now contains exactly one fewer column than the original encoded dataset ($df_{encoded.shape[1]} - 1$). This confirms that the separation was isolated strictly to the target column without any loss or duplication of predictor variables.

### Recommendation

For the storage of this dataset, I recommend using the **Apache Parquet** format.

* **Why Parquet?** Unlike CSV, Parquet is a columnar storage format that preserves your specific data types (such as `int8` for your encoded booleans) and is significantly more efficient for both disk space and I/O performance.
* **Workflow:** Save your artifacts using `X_full.to_parquet("X_full.parquet")` and `y.to_frame().to_parquet("y_full.parquet")` within your `data/processed/` directory. If you require human-readable backups for inspection, you may simultaneously save them as `.csv` files, but use the Parquet versions as your primary source for the modeling sessions to follow.

## Session 20 Prompt-Engineered Explanation Task

### Summary

The full-information feature set is **correctly separated** from the target variable. The verification results confirm that the target ($G3$) has been successfully excluded from the feature matrix ($X_{full}$), while the critical prior-grade predictors ($G1$ and $G2$) remain intact, meeting all requirements for a valid full-information modeling scenario.

### Interpretation

* **Target-Feature Separation:** By dropping the column `G3` from the feature matrix, you have effectively isolated the dependent variable. This is critical because including `G3` in the features would lead to **target leakage**—a condition where the model gains "foreknowledge" of the final result during training, leading to artificially inflated performance metrics and a failure to generalize to new, unseen data.
* **Feature Set Composition:** Your feature matrix $X_{full}$ now contains **32** features (or the total count post-encoding minus 1). The presence of $G1$ and $G2$ confirms that this is a "full-information" dataset, which allows the model to leverage a student’s academic history throughout the semester to predict the final outcome.

### Recommendation

I recommend saving your datasets in the **Apache Parquet** format.

Parquet is highly optimized for machine learning workflows because it stores data columnarly, which preserves the specific data types (such as `int8`) assigned during your feature engineering process. This ensures that when you reload your data for modeling in the next session, the memory footprint remains low and the schema is perfectly maintained. You may save the files using the `.to_parquet()` method:

```python
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")

```

If a human-readable format is required for external audits, you can also save them as `.csv` files, but use the Parquet files as the primary source for your model training.


In [63]:
# 1. Status Check
print("X_full exists:", "X_full" in globals())
print("y exists:", "y" in globals())

# 2. Generate facts for the Prompt
# Note: Ensure X_full and y are defined from previous cells before running this
feature_count = X_full.shape[1]
row_count = X_full.shape[0]
target_name = y.name
target_excluded = "G3" not in X_full.columns
g1_included = "G1" in X_full.columns
g2_included = "G2" in X_full.columns

# 3. Print Results for Documentation
print("-" * 30)
print("FACTS FOR PROMPT:")
print(f"Number of rows: {row_count}")
print(f"Number of features: {feature_count}")
print(f"Target variable: {target_name}")
print(f"G3 excluded from X_full: {target_excluded}")
print(f"G1 included in X_full: {g1_included}")
print(f"G2 included in X_full: {g2_included}")
print("-" * 30)

# 4. Final Validation Assertion
assert target_excluded == True, "❌ Target leakage detected: G3 is still present!"
assert g1_included == True and g2_included == True, "❌ G1 or G2 is missing!"
print("✅ Verification passed: Full-information dataset is prepared.")

X_full exists: True
y exists: True
------------------------------
FACTS FOR PROMPT:
Number of rows: 395
Number of features: 41
Target variable: G3
G3 excluded from X_full: True
G1 included in X_full: True
G2 included in X_full: True
------------------------------
✅ Verification passed: Full-information dataset is prepared.


Based on the GSSRP 2026 Session 20 curriculum, you are currently in the **Feature Engineering II** phase. To determine if you have completed the required steps for the "full-information feature set," check if your Google Colab environment has the following:

### 1. Verification of Objects

Ensure the following objects exist in your workspace (as per Section 3):

*
**`X_full`**: Contains your predictor variables, including $G1$ and $G2$, with the target $G3$ successfully dropped.


*
**`y`**: Contains only the target variable, $G3$.



### 2. Required Validation Steps (Section 4)

You should have already run the verification cells to confirm the following criteria are met :

*
**Target Exclusion:** The target variable $G3$ is removed from `X_full`.


*
**Predictor Inclusion:** $G1$ and $G2$ are present in `X_full`.


*
**Row Alignment:** The row counts for `X_full` and `y` match.


*
**Feature Count:** You have noted the exact feature count from `X_full.shape[1]`.



### 3. Prompt-Engineering Documentation

You must have a Text cell in your notebook labeled **"Session 20 Prompt-Engineered Explanation Task"** that contains:

1.
**Prompt Used:** The template populated with your specific feature and row counts.


2.
**AI-Generated Response:** The output you received from the AI model verifying your dataset.


3.
**Student Verification:** A statement confirming you verified the AI's response against your Python output .



### 4. GitHub Deliverable (Section 8)

Note: This is a separate step done in VS Code, not Colab.

* You must have saved your dataset as **Parquet files** (`X_full.parquet`, `y_full.parquet`, `full_information_dataset.parquet`) in your `data/processed/` folder.


* These files should be committed and pushed to your GitHub repository.



**If you have performed the Python validations in your notebook, have the prompt/response recorded, and have committed the Parquet files to GitHub, you have completed the session requirements.** If you are missing any of these, please follow the specific sections in your session materials to finalize your deliverable.

In [64]:
feature_count = X_full.shape[1]
row_count = X_full.shape[0]
target_excluded = "G3" not in X_full.columns
g1_included = "G1" in X_full.columns
g2_included = "G2" in X_full.columns

print("Number of rows:", row_count)
print("Number of features:", feature_count)
print("Target variable: G3")
print("G3 excluded from X_full:", target_excluded)
print("G1 included in X_full:", g1_included)
print("G2 included in X_full:", g2_included)

Number of rows: 395
Number of features: 41
Target variable: G3
G3 excluded from X_full: True
G1 included in X_full: True
G2 included in X_full: True


In [65]:
# Assemble Features and Target
X_full = df_encoded.drop(columns=["G3"]).copy()
y = df_encoded["G3"].copy()

# Validation Checks
assert "G3" not in X_full.columns, "Error: G3 present in features!"
assert "G1" in X_full.columns and "G2" in X_full.columns, "Error: G1/G2 missing!"
assert len(X_full) == len(y), "Error: Row mismatch!"

# Print metadata for the Prompt
print(f"Number of rows: {X_full.shape[0]}")
print(f"Number of features: {X_full.shape[1]}")
print(f"Target variable: {y.name}")
print(f"G3 excluded from X_full: {'G3' not in X_full.columns}")
print(f"G1 included in X_full: {'G1' in X_full.columns}")
print(f"G2 included in X_full: {'G2' in X_full.columns}")

Number of rows: 395
Number of features: 41
Target variable: G3
G3 excluded from X_full: True
G1 included in X_full: True
G2 included in X_full: True


In [66]:
from pathlib import Path
PROCESSED_DIR = Path("/content/drive/MyDrive/GSSRP/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save as Parquet (Required format for session 20)
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

print("✅ Parquet files saved to data/processed/")

✅ Parquet files saved to data/processed/


In [67]:
import pandas as pd
from pathlib import Path

# 1. Ensure directories exist
PROCESSED_DIR = Path("/content/drive/MyDrive/GSSRP/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. Assemble Full-Information Features and Target
# X_full includes G1 and G2; y is exclusively G3
X_full = df_encoded.drop(columns=["G3"]).copy()
y = df_encoded["G3"].copy()

# 3. Required Validation (Per Section 8 Checklist)
assert "G3" not in X_full.columns, "❌ Target leakage detected: G3 is still in features."
assert "G1" in X_full.columns and "G2" in X_full.columns, "❌ G1 or G2 missing."
assert len(X_full) == len(y), "❌ Row count mismatch."

# 4. Save to Parquet (The required format)
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

# 5. Final Output Summary
print("=" * 60)
print("SESSION 20: FULL-INFORMATION DATASET COMPLETED")
print("=" * 60)
print(f"Number of rows: {X_full.shape[0]}")
print(f"Number of features: {X_full.shape[1]}")
print(f"G3 present in X_full: {'G3' in X_full.columns}")
print(f"Parquet files saved to: {PROCESSED_DIR}")
print("=" * 60)

SESSION 20: FULL-INFORMATION DATASET COMPLETED
Number of rows: 395
Number of features: 41
G3 present in X_full: False
Parquet files saved to: /content/drive/MyDrive/GSSRP/data/processed


In [68]:
# Create the early-warning feature matrix by dropping G1 and G2
# We identify them dynamically to ensure accuracy
leak_cols = [c for c in X_full.columns if c in ("G1", "G2")]
X_early = X_full.drop(columns=leak_cols)

print("Full-information features shape:", X_full.shape)
print("Early-warning features shape:", X_early.shape)

Full-information features shape: (395, 41)
Early-warning features shape: (395, 39)


In [69]:
# Create the early-warning feature matrix
# We explicitly remove G1 and G2 to prevent "look-ahead" bias
leak_cols = [c for c in X_full.columns if c in ("G1", "G2")]
X_early = X_full.drop(columns=leak_cols)

print("Original full-information features:", X_full.shape[1])
print("Early-warning features:", X_early.shape[1])

# Verification
assert "G1" not in X_early.columns, "❌ G1 still present in X_early!"
assert "G2" not in X_early.columns, "❌ G2 still present in X_early!"
print("✅ Verification passed: Early-warning dataset is ready.")

Original full-information features: 41
Early-warning features: 39
✅ Verification passed: Early-warning dataset is ready.


In [70]:
# Save X_early and y as Parquet and CSV
X_early.to_parquet(PROCESSED_DIR / "X_early.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_early.parquet")

X_early.to_csv(PROCESSED_DIR / "X_early.csv", index=False)
y.to_csv(PROCESSED_DIR / "y_early.csv", index=False)

print("✅ Session 21 artifacts saved to data/processed/")

✅ Session 21 artifacts saved to data/processed/


In [71]:
import json

# 1. Create the Manifest (JSON)
manifest = {
    "dataset": "Early-Warning",
    "features_excluded": ["G1", "G2", "G3"],
    "total_features": int(X_early.shape[1]),
    "total_observations": int(X_early.shape[0])
}

with open(PROCESSED_DIR / "session21_early_warning_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

# 2. Create the Artifact Summary (CSV)
summary_data = {
    "File": ["X_early.parquet", "y_early.parquet", "X_early.csv", "y_early.csv"],
    "Rows": [X_early.shape[0]] * 4,
    "Columns": [X_early.shape[1], 1, X_early.shape[1], 1]
}
pd.DataFrame(summary_data).to_csv(PROCESSED_DIR / "session21_early_warning_artifact_summary.csv", index=False)

print("✅ Manifest and Summary artifacts created.")

✅ Manifest and Summary artifacts created.


In [72]:
from pathlib import Path
import pandas as pd

# Define path (ensure this matches your Drive structure)
DRIVE_BASE = Path("/content/drive/MyDrive/GSSRP")
PROCESSED_DIR = DRIVE_BASE / "data" / "processed"

# Create the directory if it's missing
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the files
X_early.to_parquet(PROCESSED_DIR / "X_early.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_early.parquet")
X_early.to_csv(PROCESSED_DIR / "X_early.csv", index=False)
y.to_csv(PROCESSED_DIR / "y_early.csv", index=False)

print(f"✅ Files saved to: {PROCESSED_DIR}")

✅ Files saved to: /content/drive/MyDrive/GSSRP/data/processed


In [74]:
# Run this in Colab to ensure they are saved to your Drive
X_early.to_parquet(PROCESSED_DIR / "X_early.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_early.parquet")
X_early.to_csv(PROCESSED_DIR / "X_early.csv", index=False)
y.to_csv(PROCESSED_DIR / "y_early.csv", index=False)